# VayuSwarm — Thermal Classifier (Kaggle)

Trains a **MobileNetV3-Small** thermal image classifier on **real infrared datasets** from Kaggle.

### Datasets
| Dataset | Kaggle slug | What it provides |
|---------|------------|-----------------|
| HIT-UAV | `pandrii000/hituav-a-highaltitude-infrared-thermal-dataset` | 2 898 real thermal IR images from UAV, YOLO bboxes: **Person, Car, OtherVehicle, Bicycle** |
| Thermal Dogs & People | `alsaniipe/thermal-dogs-and-people` | ~208 Roboflow FLIR thermal images, YOLO bboxes: **person, dog** |

### Output
`best_thermal.pth` + `thermal_classifier.onnx` → auto-pushed to `models/thermal/` in GitHub

### Kaggle Secrets required
`GIT_TOKEN`

> Runtime: ~25–35 min on T4 GPU

In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────────
import subprocess
subprocess.check_call(["pip", "install", "-q",
    "torch", "torchvision", "onnx", "onnxruntime", "onnxscript",
    "Pillow"])

import os, json, shutil, tempfile, math, zipfile

import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── CELL 2: Config & Secrets ─────────────────────────────────────────────────
GIT_TOKEN = os.environ.get("GIT_TOKEN", "")

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
except Exception:
    _s = None

if _s:
    try:
        GIT_TOKEN = _s.get_secret("GIT_TOKEN") or GIT_TOKEN
    except Exception as e:
        print(f"⚠ GIT_TOKEN secret not found: {e}")

print(f"✅ GIT_TOKEN: {'set (' + GIT_TOKEN[:8] + '…)' if GIT_TOKEN else 'EMPTY'}")

GIT_REPO  = "https://github.com/ved354/swam.git"
GIT_USER  = "ved354"
GIT_EMAIL = "ved354@users.noreply.github.com"

# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS     = 40
BATCH_SIZE = 32
LR         = 8e-4
IMG_SIZE   = 224
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "thermal_model"
HITUAV_DIR  = WORK_DIR / "hituav"               # HIT-UAV thermal dataset
TDAP_DIR    = WORK_DIR / "tdap"                  # thermal-dogs-and-people
DATASET_DIR = WORK_DIR / "thermal_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Classes that ThermalModel must predict
THERMAL_CLASSES = ["background", "human", "vehicle", "animal", "fire"]

print(f"Device : {DEVICE}")
print(f"Classes: {THERMAL_CLASSES}")

In [ ]:
# ── CELL 3: Download Datasets from Kaggle ─────────────────────────────────────
# Both datasets are public on Kaggle — no authentication needed on Kaggle notebooks.
# The `kaggle` CLI is pre-installed and pre-authenticated on Kaggle runtime.
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, zipfile

def kaggle_download(slug: str, dest: Path):
    """Download & unzip a Kaggle dataset if not already present."""
    if dest.exists() and any(dest.rglob("*")):
        print(f"  ✅ {dest.name} already exists ({sum(1 for _ in dest.rglob('*') if _.is_file())} files)")
        return
    dest.mkdir(parents=True, exist_ok=True)
    zip_name = slug.split("/")[-1] + ".zip"
    zip_path = WORK_DIR / zip_name
    print(f"  📥 Downloading {slug} …")
    subprocess.check_call([
        "kaggle", "datasets", "download", "-d", slug,
        "-p", str(WORK_DIR)
    ])
    print(f"  📦 Unzipping → {dest} …")
    with zipfile.ZipFile(str(zip_path), "r") as zf:
        zf.extractall(str(dest))
    zip_path.unlink()
    n_files = sum(1 for _ in dest.rglob("*") if _.is_file())
    print(f"  ✅ {dest.name}: {n_files} files")

# ── Also check Kaggle Input path (if user added datasets via UI) ─────────
HITUAV_INPUT = Path("/kaggle/input/hituav-a-highaltitude-infrared-thermal-dataset")
TDAP_INPUT   = Path("/kaggle/input/thermal-dogs-and-people")

if HITUAV_INPUT.exists():
    HITUAV_DIR = HITUAV_INPUT
    print(f"✅ HIT-UAV found as Kaggle Input ({sum(1 for _ in HITUAV_DIR.rglob('*') if _.is_file())} files)")
else:
    kaggle_download("pandrii000/hituav-a-highaltitude-infrared-thermal-dataset", HITUAV_DIR)

if TDAP_INPUT.exists():
    TDAP_DIR = TDAP_INPUT
    print(f"✅ Thermal Dogs & People found as Kaggle Input ({sum(1 for _ in TDAP_DIR.rglob('*') if _.is_file())} files)")
else:
    kaggle_download("alsaniipe/thermal-dogs-and-people", TDAP_DIR)

# ── Show what we have ─────────────────────────────────────────────────────────
for name, d in [("HIT-UAV", HITUAV_DIR), ("TDAP", TDAP_DIR)]:
    if d.exists():
        dirs = sorted(set(str(p.parent.relative_to(d)) for p in d.rglob("*") if p.is_file()))
        print(f"\n  {name} structure:")
        for dd in dirs[:15]:
            print(f"    {dd}")
        if len(dirs) > 15:
            print(f"    … and {len(dirs)-15} more dirs")

In [ ]:
# ── CELL 4: Extract Crops ────────────────────────────────────────────────────
import json, shutil, glob
import numpy as np
from PIL import Image

# ── HIT-UAV: YOLO format ────────────────────────────────────────────────────
# Classes in dataset.yaml: 0=Person, 1=Bicycle, 2=Car, 3=OtherVehicle
# We remap: Person→human, Car/OtherVehicle/Bicycle→vehicle

def _find_hituav_root(base: Path) -> Path:
    """Locate the actual hit-uav root (may be nested one level)."""
    # Direct structure: base/images/train/...
    if (base / "images").is_dir():
        return base
    # Nested: base/hit-uav/images/train/...
    for child in base.iterdir():
        if child.is_dir() and (child / "images").is_dir():
            return child
    return base  # fallback

def extract_hituav_crops(hituav_dir: Path, out_dir: Path, max_per_class: int = 2000):
    """
    HIT-UAV YOLO format:
      images/{train,val,test}/*.jpg
      labels/{train,val,test}/*.txt
    Each label line: class_id x_center y_center width height (normalised)
    """
    root = _find_hituav_root(hituav_dir)
    HITUAV_REMAP = {0: "human", 1: "vehicle", 2: "vehicle", 3: "vehicle"}
    counters = {"human": 0, "vehicle": 0, "background": 0}
    rng = np.random.default_rng(7)

    for split in ["train", "val", "test"]:
        img_dir   = root / "images" / split
        label_dir = root / "labels" / split
        if not img_dir.exists() or not label_dir.exists():
            continue

        img_paths = sorted(img_dir.glob("*.*"))
        for img_path in img_paths:
            label_path = label_dir / (img_path.stem + ".txt")
            try:
                img = Image.open(img_path).convert("L")
            except Exception:
                continue
            iw, ih = img.size
            ann_boxes = []   # for background extraction

            # ── Object crops ──────────────────────────────────────────────────
            if label_path.exists():
                for line in label_path.read_text().strip().splitlines():
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls_id = int(parts[0])
                    out_cls = HITUAV_REMAP.get(cls_id)
                    if out_cls is None:
                        continue
                    xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    # Convert normalised YOLO → pixel coords
                    px_w, px_h = int(w * iw), int(h * ih)
                    px_x = int(xc * iw - px_w / 2)
                    px_y = int(yc * ih - px_h / 2)
                    ann_boxes.append((px_x, px_y, px_x + px_w, px_y + px_h))

                    if counters[out_cls] >= max_per_class:
                        continue
                    pad = 10
                    crop = img.crop((max(0, px_x - pad), max(0, px_y - pad),
                                     min(iw, px_x + px_w + pad), min(ih, px_y + px_h + pad)))
                    if crop.size[0] < 12 or crop.size[1] < 12:
                        continue
                    dest = out_dir / "train" / out_cls
                    dest.mkdir(parents=True, exist_ok=True)
                    crop.resize((IMG_SIZE, IMG_SIZE)).save(
                        str(dest / f"hituav_{out_cls}_{counters[out_cls]:05d}.png"))
                    counters[out_cls] += 1

            # ── Background crops ──────────────────────────────────────────────
            if counters["background"] < max_per_class:
                bg_dest = out_dir / "train" / "background"
                bg_dest.mkdir(parents=True, exist_ok=True)
                patch = 64
                for _ in range(6):
                    if counters["background"] >= max_per_class:
                        break
                    if iw <= patch or ih <= patch:
                        break
                    cx = int(rng.integers(0, iw - patch))
                    cy = int(rng.integers(0, ih - patch))
                    overlap = any(
                        cx < x2 and cx + patch > x1 and cy < y2 and cy + patch > y1
                        for x1, y1, x2, y2 in ann_boxes
                    )
                    if overlap:
                        continue
                    crop = img.crop((cx, cy, cx + patch, cy + patch))
                    crop.resize((IMG_SIZE, IMG_SIZE)).save(
                        str(bg_dest / f"bg_{counters['background']:05d}.png"))
                    counters["background"] += 1

    for cls, n in counters.items():
        print(f"   HIT-UAV → {n:4d} {cls}")
    return counters


# ── Thermal Dogs & People: Roboflow YOLO format ─────────────────────────────
# Classes: typically 0=dog, 1=person  (from data.yaml / README)
# We remap: person→human, dog→animal

def _find_tdap_class_map(tdap_dir: Path) -> dict:
    """Auto-detect class mapping from data.yaml or README.roboflow.txt."""
    # Try data.yaml / dataset.yaml
    for yml_name in ["data.yaml", "dataset.yaml"]:
        yml = None
        for p in tdap_dir.rglob(yml_name):
            yml = p
            break
        if yml:
            try:
                import yaml
                with open(yml) as f:
                    cfg = yaml.safe_load(f)
                names = cfg.get("names", {})
                if isinstance(names, list):
                    return {i: n.lower() for i, n in enumerate(names)}
                elif isinstance(names, dict):
                    return {int(k): v.lower() for k, v in names.items()}
            except Exception:
                pass

    # Fallback: scan label files for unique class IDs
    class_ids = set()
    for lbl in tdap_dir.rglob("*.txt"):
        if lbl.name.startswith("README"):
            continue
        for line in lbl.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                class_ids.add(int(parts[0]))

    # Common Roboflow convention for this dataset: 0=dog, 1=person
    # If we only see 0 and 1, use that mapping
    if class_ids == {0, 1}:
        return {0: "dog", 1: "person"}
    elif class_ids == {0}:
        # Single class — check README
        return {0: "person"}

    # Ultimate fallback: assume person=0, dog=1
    return {i: ("person" if i == 0 else "dog") for i in sorted(class_ids)}


def extract_tdap_crops(tdap_dir: Path, out_dir: Path, max_per_class: int = 3000):
    """
    Roboflow YOLO format — images + .txt labels per folder.
    Possible layouts:
      A) train/images/*.jpg + train/labels/*.txt
      B) train/*.jpg + train/*.txt  (mixed)
    """
    TDAP_REMAP = {"person": "human", "dog": "animal", "cat": "animal"}
    class_map = _find_tdap_class_map(tdap_dir)
    print(f"   TDAP class map detected: {class_map}")
    counters = {"human": 0, "animal": 0}

    for split in ["train", "valid", "test"]:
        split_dir = None
        for p in tdap_dir.rglob(split):
            if p.is_dir():
                split_dir = p
                break
        if split_dir is None:
            continue

        # Detect layout: images/ subdirectory or mixed
        imgs_sub = split_dir / "images"
        lbls_sub = split_dir / "labels"
        if imgs_sub.is_dir():
            img_paths = sorted(imgs_sub.glob("*.*"))
            label_root = lbls_sub if lbls_sub.is_dir() else imgs_sub
        else:
            img_paths = sorted(
                p for p in split_dir.glob("*.*")
                if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
            )
            label_root = split_dir

        for img_path in img_paths:
            label_path = label_root / (img_path.stem + ".txt")
            if not label_path.exists():
                continue
            try:
                img = Image.open(img_path).convert("L")
            except Exception:
                continue
            iw, ih = img.size

            for line in label_path.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                raw_name = class_map.get(cls_id, "")
                out_cls = TDAP_REMAP.get(raw_name)
                if out_cls is None or counters.get(out_cls, 0) >= max_per_class:
                    continue
                xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                px_w, px_h = int(w * iw), int(h * ih)
                px_x = int(xc * iw - px_w / 2)
                px_y = int(yc * ih - px_h / 2)
                pad = 10
                crop = img.crop((max(0, px_x - pad), max(0, px_y - pad),
                                 min(iw, px_x + px_w + pad), min(ih, px_y + px_h + pad)))
                if crop.size[0] < 12 or crop.size[1] < 12:
                    continue
                dest = out_dir / "train" / out_cls
                dest.mkdir(parents=True, exist_ok=True)
                crop.resize((IMG_SIZE, IMG_SIZE)).save(
                    str(dest / f"tdap_{out_cls}_{counters[out_cls]:05d}.png"))
                counters[out_cls] += 1

    for cls, n in counters.items():
        print(f"   TDAP  → {n:4d} {cls}")
    return counters


def gen_fire_samples(bg_dir: Path, fire_dir: Path, n: int = 800):
    """Simulate saturated fire signatures on real thermal backgrounds."""
    fire_dir.mkdir(parents=True, exist_ok=True)
    bgs = list(bg_dir.glob("*.png"))[:300] if bg_dir.exists() else []
    rng = np.random.default_rng(42)
    for i in range(n):
        if bgs:
            base = np.array(Image.open(bgs[i % len(bgs)]).convert("L")
                            .resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)
        else:
            base = rng.normal(35, 8, (IMG_SIZE, IMG_SIZE)).clip(0, 80).astype(np.float32)
        n_blobs = rng.integers(1, 4)
        for _ in range(n_blobs):
            cx = rng.integers(25, IMG_SIZE - 25)
            cy = rng.integers(25, IMG_SIZE - 25)
            r  = rng.integers(6, 28)
            ys, xs = np.ogrid[-r:r+1, -r:r+1]
            mask = (xs**2 + ys**2) <= r**2
            py = np.clip(cy + np.arange(-r, r+1)[:, None], 0, IMG_SIZE - 1)
            px = np.clip(cx + np.arange(-r, r+1)[None, :], 0, IMG_SIZE - 1)
            intensity = 255 - (25 * np.sqrt(xs**2 + ys**2) / r)
            base[py[mask], px[mask]] = np.maximum(base[py[mask], px[mask]], intensity[mask])
        Image.fromarray(base.clip(0, 255).astype(np.uint8), mode="L").save(
            str(fire_dir / f"fire_{i:05d}.png"))
    print(f"   Fire  → {n:4d} physics-accurate samples")


# ── Run extraction ────────────────────────────────────────────────────────────
print("\n📦 Building thermal dataset from real images...")
extract_hituav_crops(HITUAV_DIR, DATASET_DIR, max_per_class=2000)
extract_tdap_crops(TDAP_DIR, DATASET_DIR, max_per_class=3000)
gen_fire_samples(DATASET_DIR / "train" / "background",
                 DATASET_DIR / "train" / "fire", n=800)

# ── Val split (20%) ───────────────────────────────────────────────────────────
rng = np.random.default_rng(0)
print("\n🔀 Creating val split (20%)...")
for cls in THERMAL_CLASSES:
    src = DATASET_DIR / "train" / cls
    val = DATASET_DIR / "val"   / cls
    val.mkdir(parents=True, exist_ok=True)
    imgs = list(src.glob("*")) if src.exists() else []
    rng.shuffle(imgs)
    for p in imgs[:max(30, len(imgs) // 5)]:
        shutil.copy2(str(p), str(val / p.name))

print("\n📊 Class sizes (train):")
for cls in THERMAL_CLASSES:
    n = len(list((DATASET_DIR / "train" / cls).glob("*"))) if \
        (DATASET_DIR / "train" / cls).exists() else 0
    print(f"   {cls:12s}: {n}")

In [ ]:
# ── CELL 5: Dataset & DataLoaders ────────────────────────────────────────────
class ThermalDataset(Dataset):
    def __init__(self, root: Path, transform):
        self.samples   = []
        self.transform = transform
        for ci, cls in enumerate(THERMAL_CLASSES):
            d = root / cls
            if d.exists():
                for p in d.glob("*"):
                    self.samples.append((str(p), ci))
        np.random.default_rng(42).shuffle(self.samples)

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        img = Image.open(self.samples[i][0]).convert("L")
        return self.transform(img), self.samples[i][1]


# Augmentation — thermal-specific (no colour jitter on grayscale)
t_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),      # L → RGB for MobileNet
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.1)), # simulate sensor noise
])
t_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = ThermalDataset(DATASET_DIR / "train", t_train)
val_ds   = ThermalDataset(DATASET_DIR / "val",   t_val)
print(f"Dataset: {len(train_ds)} train  {len(val_ds)} val  (real thermal images)")

# Class-balanced sampler weights
counts  = [max(1, len(list((DATASET_DIR/"train"/c).glob("*")))) for c in THERMAL_CLASSES]
cls_wts = torch.tensor([sum(counts)/c for c in counts], dtype=torch.float).to(DEVICE)
print("Class weights:", {c: f"{w:.2f}" for c, w in zip(THERMAL_CLASSES, cls_wts.tolist())})

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)


# ── CELL 5b: Model — MobileNetV3-Small (matches VayuSwarm inference code) ────
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(THERMAL_CLASSES))
model = model.to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MobileNetV3-Small: {params:,} parameters, {len(THERMAL_CLASSES)} classes")

In [ ]:
# ── CELL 6: Training Loop ────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=cls_wts)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
no_improve = 0
PATIENCE = 10

print(f"🚀 Training {EPOCHS} epochs on real HIT-UAV + TDAP data...")

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    tc = tt = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)              # single forward pass — reused for both loss and acc
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        _, pred = out.max(1)           # use same output, no second forward pass
        tt += labels.size(0)
        tc += pred.eq(labels).sum().item()
    scheduler.step()

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    vc = vt = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            _, pred = model(imgs).max(1)
            vt += labels.size(0)
            vc += pred.eq(labels).sum().item()

    va = 100.0 * vc / vt
    print(f"Epoch {epoch+1:3d}/{EPOCHS}  train={100.*tc/tt:.1f}%  val={va:.1f}%")

    if va > best_acc:
        best_acc = va
        no_improve = 0
        torch.save(model.state_dict(), str(OUTPUT_DIR / "best_thermal.pth"))
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"⏹ Early stop at epoch {epoch+1}")
            break

print(f"\n✅ Best val accuracy: {best_acc:.1f}% (real thermal images)")

In [ ]:
# ── CELL 7: Export ONNX + Save Metadata ─────────────────────────────────────
model.load_state_dict(torch.load(str(OUTPUT_DIR / "best_thermal.pth"), weights_only=True))
model.eval()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
torch.onnx.export(
    model, dummy,
    str(OUTPUT_DIR / "thermal_classifier.onnx"),
    input_names=["thermal_image"],
    output_names=["class_probs"],
    dynamic_axes={"thermal_image": {0: "batch"}, "class_probs": {0: "batch"}},
    opset_version=17,
)

metadata = {
    "model":      "vayuswarm_thermal",
    "classes":    THERMAL_CLASSES,
    "input_size": IMG_SIZE,
    "best_val_acc": round(best_acc, 2),
    "training_data": {
        "human":      "HIT-UAV (Person) + Thermal Dogs & People (person)",
        "vehicle":    "HIT-UAV (Car, OtherVehicle, Bicycle)",
        "animal":     "Thermal Dogs & People (dog)",
        "background": "HIT-UAV background patches",
        "fire":       "Physics-accurate fire signatures on real IR backgrounds",
    },
    "datasets": [
        "pandrii000/hituav-a-highaltitude-infrared-thermal-dataset",
        "alsaniipe/thermal-dogs-and-people",
    ],
    "train_samples": len(train_ds),
    "val_samples":   len(val_ds),
}
with open(OUTPUT_DIR / "thermal_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Exported: best_thermal.pth  thermal_classifier.onnx  thermal_metadata.json")
print(f"   Val accuracy: {best_acc:.1f}%")
print(f"   Classes: {THERMAL_CLASSES}")

In [ ]:
# ── CELL 8: Push to GitHub ───────────────────────────────────────────────────
import subprocess as _sp

if GIT_TOKEN:
    try:
        auth_url  = GIT_REPO.replace("https://", f"https://{GIT_USER}:{GIT_TOKEN}@")
        clone_dir = Path(tempfile.mkdtemp()) / "swam"
        print(f"\n📤 Cloning {GIT_REPO}...")
        _sp.check_call(["git", "clone", "--depth", "1", auth_url, str(clone_dir)])

        target = clone_dir / "models" / "thermal"
        target.mkdir(parents=True, exist_ok=True)

        for fname in ["best_thermal.pth", "thermal_classifier.onnx", "thermal_metadata.json"]:
            src = OUTPUT_DIR / fname
            if src.exists():
                shutil.copy2(str(src), str(target / fname))
                print(f"   ✅ {fname} ({src.stat().st_size / 1024 / 1024:.1f} MB)")

        env = os.environ.copy()
        for cmd in [
            ["git", "config", "user.name",  GIT_USER],
            ["git", "config", "user.email", GIT_EMAIL],
            ["git", "add", "models/thermal/"],
            ["git", "commit", "-m",
             f"Real-data thermal classifier — val_acc={best_acc:.1f}%, "
             f"datasets: HIT-UAV+TDAP, {len(THERMAL_CLASSES)} classes"],
            ["git", "push", "origin", "main"],
        ]:
            _sp.check_call(cmd, cwd=str(clone_dir), env=env)

        print(f"\n✅ Pushed thermal model to {GIT_REPO}")

    except Exception as e:
        print(f"\n⚠ GitHub push failed: {e}")
        print("  → Download from Kaggle Output tab: thermal_model/")
else:
    print("ℹ No GIT_TOKEN — model saved locally at:", OUTPUT_DIR)

print(f"""
{'='*55}
🎉 Thermal Training Complete!
   Val accuracy : {best_acc:.1f}%
   Datasets     : HIT-UAV + Thermal Dogs & People
   Classes      : {THERMAL_CLASSES}
{'='*55}
""")